In [25]:
import os
import sys
import subprocess

conda_path = "/opt/homebrew/Caskroom/miniconda/base/envs/proGAT"

In [3]:
dir_datas = "/Users/latterday/Desktop/Project/proGAT/datas"

In [16]:
file_temp = "/Users/latterday/Desktop/Project/proGAT/file_temp"
os.makedirs(file_temp, exist_ok=True)

In [7]:
if os.path.exists(dir_datas)==False:
    print(f"数据目录 {dir_datas} 不存在，请检查路径。")
    sys.exit(1)

In [9]:
sample_id = os.listdir(dir_datas)

In [10]:
sample_id

['SRR23100674.fastq.gz', 'SRR23100672.fastq.gz']

seqkit 统计

In [29]:
temp_seqkit = os.path.join(file_temp, "seqkit_stats")
os.makedirs(temp_seqkit, exist_ok=True)

for sample in sample_id:
    temp_file = os.path.join(dir_datas, sample)
    sample_name = sample
    for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq"):
        if sample_name.endswith(suffix):
            sample_name = sample_name.removesuffix(suffix)
            break
    
    subprocess.run(f"seqkit stats {temp_file}  -a -T > {temp_seqkit}/{sample_name}_stats.txt", shell=True)

fastplong 过滤

In [28]:
fastplong_bin = os.path.join(conda_path, "bin", "fastplong")

temp_fastplong = os.path.join(file_temp, "fastplong_filtered")
os.makedirs(temp_fastplong, exist_ok=True)

for sample in sample_id:
    input_file = os.path.join(dir_datas, sample)

    # 去除常见的 FASTQ 扩展名，得到干净的样本名称
    sample_name = sample
    for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq"):
        if sample_name.endswith(suffix):
            sample_name = sample_name.removesuffix(suffix)
            break

    # 每个样本单独建立目录
    sample_output_dir = os.path.join(
        temp_fastplong,
        sample_name,
    )
    os.makedirs(sample_output_dir, exist_ok=True)

    output_fastq = os.path.join(
        sample_output_dir,
        f"{sample_name}_filtered.fastq.gz",
    )
    output_html = os.path.join(
        sample_output_dir,
        f"{sample_name}_fastplong.html",
    )
    output_json = os.path.join(
        sample_output_dir,
        f"{sample_name}_fastplong.json",
    )

    subprocess.run(
        [
            fastplong_bin,
            "-i", input_file,
            "-o", output_fastq,
            "-h", output_html,
            "-j", output_json,
        ],
        check=True,
    )

Trying to detect adapter sequence at read start
Not detected
Trying to detect adapter sequence at read end
Found possible adapter sequence, but it's too short: AGGTGCTGCAGGTA, specify -e AGGTGCTGCAGGTA to force trimming using this adapter

Before filtering:
total reads: 251505
total bases: 437387806
Q20 bases: 355258897(81.2229%)
Q30 bases: 284789508(65.1114%)

After filtering:
total reads: 241911
total bases: 419736197
Q20 bases: 347635869(82.8225%)
Q30 bases: 280672497(66.8688%)

Filtering result:
reads passed filter: 241911
reads failed due to low quality: 9594
reads failed due to too many N: 0
reads failed due to too short: 0
reads with adapter trimmed: 0
bases trimmed due to adapters: 0

JSON report: /Users/latterday/Desktop/Project/proGAT/file_temp/fastplong_filtered/SRR23100674/SRR23100674_fastplong.json
HTML report: /Users/latterday/Desktop/Project/proGAT/file_temp/fastplong_filtered/SRR23100674/SRR23100674_fastplong.html

/opt/homebrew/Caskroom/miniconda/base/envs/proGAT/bin/f

lrge 基因组大小预估

In [38]:
temp_lrge = os.path.join(file_temp, "lrge")
os.makedirs(temp_lrge, exist_ok=True)
docker_image = "staphb/lrge:latest"


for sample in sample_id:
    
    sample_name = os.path.basename(sample)
    for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq"):
        if sample_name.endswith(suffix):
            sample_name = sample_name.removesuffix(suffix)
            break
    
    input_file = os.path.join(temp_fastplong, sample_name, f"{sample_name}_filtered.fastq.gz")
    
    output_name = f"{sample_name}_size.txt"
    
    subprocess.run(
        [
            "docker", "run", "--rm",
            "--platform", "linux/amd64",
            "-v", f"{input_file}:/input.fastq.gz:ro",
            "-v", f"{temp_lrge}:/output",
            docker_image,
            "lrge",
            "-P", "ont",
            "-t", "8",
            "-o", f"/output/{output_name}",
            "/input.fastq.gz",
        ],
        check=True,
    )
    

[2026-08-13T09:25:26Z INFO  lrge] Running two-set strategy with 10000 target reads and 5000 query reads
[2026-08-13T09:25:49Z INFO  liblrge::twoset] 249 (4.98%) query read(s) did not overlap any target reads
[2026-08-13T09:25:49Z INFO  lrge] Estimated genome size: 5.45 Mbp (IQR: 3.43 Mbp - 6.47 Mbp)
[2026-08-13T09:25:49Z INFO  lrge] Done!
[2026-08-13T09:25:50Z INFO  lrge] Running two-set strategy with 10000 target reads and 5000 query reads
[2026-08-13T09:25:57Z INFO  liblrge::twoset] 262 (5.24%) query read(s) did not overlap any target reads
[2026-08-13T09:25:57Z INFO  lrge] Estimated genome size: 5.46 Mbp (IQR: 3.30 Mbp - 6.21 Mbp)
[2026-08-13T09:25:57Z INFO  lrge] Done!


In [37]:
temp_fastplong

'/Users/latterday/Desktop/Project/proGAT/file_temp/fastplong_filtered'

flye拼接

In [ ]:
temp_flye = os.path.join(file_temp, "flye")
os.makedirs(temp_flye, exist_ok=True)

for sample in sample_id:
    sample_name = os.path.basename(sample)
    for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq"):
        if sample_name.endswith(suffix):
            sample_name = sample_name.removesuffix(suffix)
            break
    
    input_file = os.path.join(temp_fastplong, sample_name, f"{sample_name}_filtered.fastq.gz")
    
    file_lrge = os.path.join(temp_lrge, f"{sample_name}_size.txt")
    
    subprocess.run([
        "flye",
        "--nano-hq", input_file,
        "--genome-size", "5m",
        "--out-dir", os.path.join(temp_flye, sample_name),
        "--threads", "8"
    ], check=True)


[2026-08-13 17:27:18] INFO: Starting Flye 2.9.6-b1802
[2026-08-13 17:27:18] INFO: >>>STAGE: configure
[2026-08-13 17:27:18] INFO: Configuring run
[2026-08-13 17:27:20] INFO: Total read length: 419736197
[2026-08-13 17:27:20] INFO: Input genome size: 5000000
[2026-08-13 17:27:20] INFO: Estimated coverage: 83
[2026-08-13 17:27:20] INFO: Reads N50/N90: 5003 / 584
[2026-08-13 17:27:20] INFO: Minimum overlap set to 1000
[2026-08-13 17:27:20] INFO: >>>STAGE: assembly
[2026-08-13 17:27:20] INFO: Assembling disjointigs
[2026-08-13 17:27:20] INFO: Reading sequences
[2026-08-13 17:27:23] INFO: Building minimizer index
[2026-08-13 17:27:23] INFO: Pre-calculating index storage
0% 10% 20% 30% 40% 50% 60% 70% 80% 90% 100% 
[2026-08-13 17:27:26] INFO: Filling index
0% 10% 20% 30% 40% 50% 60% 70% 80% 90% 100% 
[2026-08-13 17:27:31] INFO: Extending reads
[2026-08-13 17:27:43] INFO: Overlap-based coverage: 50
[2026-08-13 17:27:43] INFO: Median overlap divergence: 0.0336591
0% 80% 100% 
[2026-08-13 17:28